In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('data/processed/churn_cleaned.csv')

# Fix TotalCharges — may contain empty strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# 1. Encode categorical variables
le_dict = {}
all_cat = ['gender', 'Partner', 'Dependents', 'PhoneService',
           'InternetService', 'OnlineSecurity', 'Contract', 'PaymentMethod']
categorical_cols = [c for c in all_cat if c in df.columns and df[c].dtype == 'object']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

# 2. Create new features
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=[0, 1, 2, 3])
df['MonthlyChargesGroup'] = pd.qcut(df['MonthlyCharges'], q=4, labels=[0, 1, 2, 3])
df['TotalChargesPerMonth'] = df['TotalCharges'] / (df['tenure'] + 1)

# 3. Check correlation with churn
correlations = df.corr(numeric_only=True)['Churn'].sort_values(ascending=False)
print(correlations)

# 4. Feature scaling — define numeric_cols first
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalChargesPerMonth']
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df_scaled.to_csv('data/processed/churn_engineered.csv', index=False)
print('\n✅ Feature engineering complete! Saved to data/processed/churn_engineered.csv')


Churn                   1.000000
MonthlyCharges          0.193356
SeniorCitizen           0.150889
PaymentMethod           0.107062
TotalChargesPerMonth    0.014873
PhoneService            0.011942
gender                 -0.008612
InternetService        -0.047291
Partner                -0.150448
Dependents             -0.164221
TotalCharges           -0.199037
OnlineSecurity         -0.289309
tenure                 -0.352229
Contract               -0.396713
Name: Churn, dtype: float64

✅ Feature engineering complete! Saved to data/processed/churn_engineered.csv
